# Phòng thí nghiệm QR Lotto 5/35 (Colab)

Chạy trong Colab của chính bạn sau khi tải toàn bộ dự án `xstk` lên môi trường Colab (Files → Upload, hoặc Drive tùy chọn). Không tự kết nối tài khoản, không tải dữ liệu từ mạng. Không mô phỏng mã RNG OKC; thí nghiệm không tạo lợi thế trúng thưởng. Các ô đánh dấu tùy chọn cần bạn chủ động bật.


In [ ]:
%pip install numpy torch


## Chọn thư mục dữ liệu
Tải thư mục dự án (bao gồm `rng_lab.py`, `rng_forecast.py`, `rng_neural.py`, `vietlott_535.json`) vào Colab trước khi chạy; ô dưới kiểm tra sự tồn tại thực tế, không giả định rằng dữ liệu đã tải lên. Tùy chọn Drive được tắt mặc định và cần bạn tự xác định thư mục.


In [ ]:
from pathlib import Path
USE_DRIVE = False  # Bật thủ công nếu bạn muốn gắn Google Drive.
DRIVE_REPO = None  # Nếu USE_DRIVE, tự gán chuỗi đường dẫn thư mục dự án trên Drive.
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')  # Colab yêu cầu người dùng xác thực.
    if DRIVE_REPO is None:
        raise ValueError('Gán DRIVE_REPO tới thư mục dự án của bạn trên Drive.')
REPO = Path(DRIVE_REPO) if USE_DRIVE else (Path.cwd() / 'xstk' if (Path.cwd() / 'xstk' / 'rng_lab.py').is_file() else Path.cwd())
needed = ['rng_lab.py', 'rng_forecast.py', 'rng_neural.py', 'qr_filter.py', 'qr_backtest.py', 'qr_predict.py', 'vietlott_535.json']
missing = [name for name in needed if not (REPO / name).is_file()]
if missing:
    raise FileNotFoundError(f'Tải toàn bộ dự án lên Colab hoặc chỉnh REPO tới thư mục dự án; thiếu: {missing}; đã kiểm tra: {REPO}')
print('Thư mục đã kiểm tra:', REPO.resolve())
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## Giai đoạn 1 — tham chiếu toán học chính xác
Tất cả 324632 tổ hợp 5 số được liệt kê và so với QR80; CPU đủ dùng.


In [ ]:
import subprocess, sys
def lab(*args):
    return subprocess.run([sys.executable, 'rng_lab.py', *map(str, args)], cwd=REPO, check=True)
lab('reference')


## Giai đoạn 2 — kiểm tra lịch sử và mô phỏng công suất B1–B5
Các phép thử có seed cố định; kiểm soát lỗi nhiều kiểm định. Công suất chỉ có nghĩa cho các giả thuyết lỗi đã định nghĩa, không chứng minh máy quay thật bị lệch. Bật chạy công suất riêng vì tốn thời gian.


In [ ]:
lab('audit', '--source', REPO / 'vietlott_535.json', '--repetitions', 500, '--seed', 20261029)
RUN_POWER = False
if RUN_POWER:
    lab('power', '--lengths', 250, 976, 2000, '--strengths', .02, .1, .3, '--repetitions', 500, '--seed', 20261029)


## Giai đoạn 3 — luồng giả lập đồng đều, dự báo theo thời gian và khóa tiền tố
Bản mô phỏng là NumPy/PCG64, không phải OKC. Mặc định chạy 1 triệu kỳ theo batch, có thể tăng lên 1 tỷ nếu tự đánh giá thời gian và tài nguyên. Báo cáo hồi cứu không đủ mở cổng neural. Khóa manifest trước khi có kỳ tương lai: chỉ bật một lần với lịch sử hiện tại, sau này dùng chính manifest cũ để chấm những kỳ mới. Không khóa và chấm cùng dữ liệu để tự nhận là tiến cứu.


In [ ]:
lab('simulate', '--draws', 1_000_000, '--batch-size', 100_000, '--model', 'uniform', '--seed', 20261029)
lab('forecast', '--source', REPO / 'vietlott_535.json')  # hồi cứu, không phải bằng chứng
LOCK = REPO / 'forecast_frozen_manifest.json'
PROSPECTIVE_REPORT = REPO / 'forecast_prospective.json'
CREATE_LOCK_NOW = False  # Chỉ bật TRƯỚC khi có dữ liệu mới; không ghi đè manifest cũ.
if CREATE_LOCK_NOW:
    lab('forecast', '--source', REPO / 'vietlott_535.json', '--lock', LOCK)
if LOCK.is_file():
    if PROSPECTIVE_REPORT.exists():
        print('Báo cáo cũ đã tồn tại; chỉ tạo tên báo cáo mới khi có dữ liệu kỳ mới.')
    else:
        lab('forecast', '--source', REPO / 'vietlott_535.json', '--prospective', LOCK, '--output', PROSPECTIVE_REPORT)
else:
    print('Chưa có manifest đã khóa; không thể chấm tiến cứu.')


## Giai đoạn 4 — neural có cổng bằng chứng
Chạy mặc định luôn: nếu thiếu báo cáo tiến cứu với ≥100 kỳ MỚI và e-value ≥20 từ mô hình đã khóa, lệnh trả `deferred`, **không tạo checkpoint**, không cần import torch. `--diagnostic` là kiểm tra kỹ thuật hồi cứu hoặc giả lập độc lập, không phải tín hiệu hay lợi thế; chỉ bật chủ động. Mô hình gán xác suất dương cho mọi 5+1 (softmax khối QR / số bộ chính trong khối × softmax 12 số đặc biệt), chia train/validation/test theo thời gian.


In [ ]:
neural_args = ['neural', '--source', REPO / 'vietlott_535.json', '--device', 'auto']
if PROSPECTIVE_REPORT.is_file():
    neural_args += ['--prospective-report', PROSPECTIVE_REPORT]
lab(*neural_args)
RUN_DIAGNOSTIC = False  # Chỉ kiểm thử kỹ thuật; file mới và duy nhất mỗi lần chạy.
if RUN_DIAGNOSTIC:
    lab('neural', '--diagnostic', '--synthetic-draws', 200, '--device', 'auto', '--epochs', 3, '--seed', 535, '--checkpoint', REPO / 'rng_neural_diagnostic.pt')


## Giai đoạn 5 — tra cứu một vé
Báo cáo xác suất công bằng, đặc trưng và xác suất *được mô hình gán*; không phải xác suất trúng thực tế khi quay công bằng.


In [ ]:
lab('ticket', 6, 15, 21, 30, 34, 6, '--source', REPO / 'vietlott_535.json')
